In [4]:
import cv2
import numpy as np
import mediapipe as mp
import trimesh
import pyrender

# ===== app.py content =====
# Load mesh once
MESH_PATH = "model.obj"  # Update path as needed
shoe_trimesh = trimesh.load(MESH_PATH, force='mesh')

# Scale if needed (e.g., mm to meters)
# shoe_trimesh.apply_scale(0.001)

# Create mesh and light
shoe_mesh = pyrender.Mesh.from_trimesh(shoe_trimesh, smooth=False)
light = pyrender.DirectionalLight(color=np.ones(3), intensity=3.0)

def make_base_scene(fx, fy, cx, cy):
    scene = pyrender.Scene(ambient_light=[0.2, 0.2, 0.2])
    cam = pyrender.IntrinsicsCamera(fx, fy, cx, cy)
    cam_node = scene.add(cam, pose=np.eye(4))
    light_pose = np.array([[1, 0, 0, 0],
                          [0, 1, 0, 0],
                          [0, 0, 1, 4],
                          [0, 0, 0, 1]], dtype=np.float32)
    light_node = scene.add(light, pose=light_pose)
    shoe_node = scene.add(shoe_mesh, pose=np.eye(4))
    return scene, shoe_node

def alpha_composite(background_bgr, render_rgba):
    rgb = render_rgba[..., :3]
    a = render_rgba[..., 3:4] / 255.0
    rgb_bgr = rgb[..., ::-1]
    out = background_bgr.astype(np.float32) * (1 - a) + rgb_bgr.astype(np.float32) * a
    return out.astype(np.uint8)

# OpenCV to OpenGL coordinate system transformation
T_CV2GL = np.array([[1, 0, 0, 0],
                    [0, -1, 0, 0],
                    [0, 0, -1, 0],
                    [0, 0, 0, 1]], dtype=np.float32)

def render_on_frame(frame_bgr, foot_transform_cv, fx, fy, cx, cy, renderer_cache={}):
    H, W = frame_bgr.shape[:2]
    key = (W, H, fx, fy, cx, cy)
    
    if key not in renderer_cache:
        scene, shoe_node = make_base_scene(fx, fy, cx, cy)
        renderer = pyrender.OffscreenRenderer(viewport_width=W, viewport_height=H)
        renderer_cache[key] = (scene, shoe_node, renderer)
    else:
        scene, shoe_node, renderer = renderer_cache[key]
    
    # Apply coordinate system transformation
    pose_gl = T_CV2GL @ foot_transform_cv
    scene.set_pose(shoe_node, pose=pose_gl)
    
    color_rgba, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
    return alpha_composite(frame_bgr, color_rgba)

ValueError: string is not a file: `model.obj`

In [3]:
!pip install trimesh pyrender mediapipe opencv-python


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.9 MB/s eta 0:00:00 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 709.0/709.0 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 38.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.0/984.0 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.7 MB/s eta 0:00:00
  Created wheel for PyOpenGL: filename=pyopengl-3.1.0-py3-none-any.whl size=1745237 sha256=fd2a5f9e631f6f9a642be86eec03b439c9baf1e392c1b4fc3efb983e